# Qwen 1.5B Fine-Tuning Notebook

Lucas Harrich

This is a simple finetune approach with Qwen 1.5B.

It uses your training data from `finetune_data/austrian_corp_tax_seed_sft.jsonl`, fine-tunes the last model layers directly, and then writes a new submission CSV.

What it does:
- it takes all rows from `finetune_data/austrian_corp_tax_seed_sft.jsonl`
- it writes `train.jsonl` for MLX-LM
- it fine-tunes `Qwen/Qwen2.5-1.5B-Instruct`
- it saves the trained output to `qwen_1_5b_tax_fullft_mlx_full`
- it writes `finetune_awen_1_5b.csv` at the end


In [6]:
import subprocess
import sys

packageList = ["mlx-lm[train]", "transformers"]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *packageList])


Python(47151) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


0

In [ ]:
# Imports and settings
import csv
import json
import random
import subprocess
import sys
from datetime import datetime as datetimeClass
from pathlib import Path as pathClass

from mlx_lm import generate as generateTextFunc
from mlx_lm import load as loadModelFunc
from mlx_lm.sample_utils import make_logits_processors as makeLogitsProcessorsFunc
from mlx_lm.sample_utils import make_sampler as makeSamplerFunc

inputPath = pathClass("training_data/austrian_corp_tax_seed_sft.jsonl")
mlxDataPath = pathClass("training_data/mlx_qwen_1_5b_fullft")
baseOutputPath = pathClass("qwen_1_5b_tax_fullft_mlx_full")
outputPath = baseOutputPath
predictionInputPath = pathClass("dataset_clean.csv")

randomSeed = 42
modelName = "Qwen/Qwen2.5-1.5B-Instruct"
maxSequenceLength = 300
maxNewTokens = 130

trainBatchSize = 1
gradientAccumulationSteps = 4
learningRate = 2e-6
numLayers = 2
trainPasses = 2
stepsPerSave = None

generationTemperature = 0.7
generationTopP = 0.9
generationTopK = 40
repetitionPenalty = 1.08
repetitionContextSize = 48

usePredictionSample = False
predictionSampleSize = 5

systemPrompt = (
    "Beantworte die oesterreichische Steuerfrage auf Deutsch "
    "Sei konkret. Wiederhole die Frage nicht. Erfinde keine Fakten."
)

requiredColumns = {"messages"}
requiredPredictionColumns = {"id", "prompt"}


: 

In [8]:
# Prepare MLX-LM training data
rowList = []
with inputPath.open("r", encoding="utf-8") as sourceFile:
    for line in sourceFile:
        if not line.strip():
            continue

        row = json.loads(line)
        missingColumns = requiredColumns - set(row)
        if missingColumns:
            raise ValueError(f"Missing columns: {sorted(missingColumns)}")

        rowList.append({"messages": row["messages"]})

if not rowList:
    raise ValueError("Training file is empty.")

random.Random(randomSeed).shuffle(rowList)

mlxDataPath.mkdir(parents=True, exist_ok=True)
trainPath = mlxDataPath / "train.jsonl"

with trainPath.open("w", encoding="utf-8") as targetFile:
    for row in rowList:
        targetFile.write(json.dumps(row, ensure_ascii=False) + "\n")

trainIterations = len(rowList) * trainPasses
checkpointSaveEvery = stepsPerSave or (trainIterations + 1)

if outputPath.exists() and any(outputPath.iterdir()):
    runTimestamp = datetimeClass.now().strftime("%Y%m%d_%H%M%S")
    outputPath = baseOutputPath.with_name(f"{baseOutputPath.name}_{runTimestamp}")

outputPath.mkdir(parents=True, exist_ok=True)

print(f"rows: {len(rowList)}")
print(f"train file: {trainPath.resolve()}")
print(f"train iterations: {trainIterations}")
print(f"train output: {outputPath.resolve()}")
print(f"save every: {checkpointSaveEvery}")
print(f"model: {modelName}")
print(f"num layers: {numLayers}")
print(f"learning rate: {learningRate}")


rows: 146
train file: /Users/lucas/Documents/Data Applications/training_data/mlx_qwen_1_5b_fullft/train.jsonl
train iterations: 292
train output: /Users/lucas/Documents/Data Applications/qwen_1_5b_tax_fullft_mlx_full_20260407_094615
save every: 293
model: Qwen/Qwen2.5-1.5B-Instruct
num layers: 2
learning rate: 2e-06


In [9]:
# Full fine-tuning of the last layers with MLX-LM
commandList = [
    sys.executable,
    "-m",
    "mlx_lm",
    "lora",
    "--model",
    modelName,
    "--train",
    "--fine-tune-type",
    "full",
    "--data",
    str(mlxDataPath),
    "--adapter-path",
    str(outputPath),
    "--iters",
    str(trainIterations),
    "--batch-size",
    str(trainBatchSize),
    "--grad-accumulation-steps",
    str(gradientAccumulationSteps),
    "--learning-rate",
    str(learningRate),
    "--num-layers",
    str(numLayers),
    "--max-seq-length",
    str(maxSequenceLength),
    "--steps-per-report",
    "25",
    "--save-every",
    str(checkpointSaveEvery),
    "--mask-prompt",
    "--seed",
    str(randomSeed),
]

print(" ".join(commandList))
subprocess.run(commandList, check=True)
print(f"saved trained output: {outputPath.resolve()}")


/Users/lucas/Documents/Data Applications/.venv/bin/python -m mlx_lm lora --model Qwen/Qwen2.5-1.5B-Instruct --train --fine-tune-type full --data training_data/mlx_qwen_1_5b_fullft --adapter-path qwen_1_5b_tax_fullft_mlx_full_20260407_094615 --iters 292 --batch-size 1 --grad-accumulation-steps 4 --learning-rate 2e-06 --num-layers 2 --max-seq-length 300 --steps-per-report 25 --save-every 293 --mask-prompt --seed 42


Python(47157) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


Loading pretrained model


Fetching 7 files: 100%|██████████| 7/7 [00:00<00:00, 81329.99it/s]


Loading datasets
Training
Trainable parameters: 6.063% (93.596M/1543.714M)
Starting training..., iters: 292
Iter 25: Train loss 2.493, Learning Rate 2.000e-06, It/sec 0.423, Tokens/sec 17.321, Trained Tokens 1024, Peak mem 4.494 GB
Iter 50: Train loss 2.357, Learning Rate 2.000e-06, It/sec 0.963, Tokens/sec 44.645, Trained Tokens 2183, Peak mem 4.510 GB
Iter 75: Train loss 2.296, Learning Rate 2.000e-06, It/sec 1.674, Tokens/sec 75.919, Trained Tokens 3317, Peak mem 4.510 GB
Iter 100: Train loss 2.139, Learning Rate 2.000e-06, It/sec 0.235, Tokens/sec 10.384, Trained Tokens 4420, Peak mem 4.510 GB
Iter 125: Train loss 1.791, Learning Rate 2.000e-06, It/sec 2.328, Tokens/sec 102.259, Trained Tokens 5518, Peak mem 4.510 GB
Iter 150: Train loss 1.803, Learning Rate 2.000e-06, It/sec 0.266, Tokens/sec 12.044, Trained Tokens 6651, Peak mem 4.510 GB
Iter 175: Train loss 1.779, Learning Rate 2.000e-06, It/sec 2.768, Tokens/sec 129.213, Trained Tokens 7818, Peak mem 4.510 GB
Iter 200: Train lo

In [10]:
# Create a submission CSV with sampling and repetition control
model, tokenizer = loadModelFunc(modelName, adapter_path=str(outputPath))

sampler = makeSamplerFunc(
    temp=generationTemperature,
    top_p=generationTopP,
    top_k=generationTopK,
)

logitsProcessors = makeLogitsProcessorsFunc(
    repetition_penalty=repetitionPenalty,
    repetition_context_size=repetitionContextSize,
)

def cleanAnswer(text):
    compactText = " ".join(str(text).split()).strip()
    if not compactText:
        return compactText

    sentenceList = []
    for piece in compactText.replace("!", ".").replace("?", ".").split("."):
        piece = piece.strip()
        if piece:
            sentenceList.append(piece)

    uniqueSentenceList = []
    seenSet = set()
    for sentence in sentenceList:
        key = sentence.lower()
        if key not in seenSet:
            seenSet.add(key)
            uniqueSentenceList.append(sentence)

    cleanedText = ". ".join(uniqueSentenceList[:3]).strip()
    if cleanedText and cleanedText[-1] not in ".!?":
        cleanedText += "."
    return cleanedText

with predictionInputPath.open("r", encoding="utf-8-sig", newline="") as sourceFile:
    reader = csv.DictReader(sourceFile)
    missingColumns = requiredPredictionColumns - set(reader.fieldnames or [])
    if missingColumns:
        raise ValueError(f"Missing columns: {sorted(missingColumns)}")

    rowList = list(reader)

if usePredictionSample:
    rowList = rowList[:predictionSampleSize]

with predictionOutputPath.open("w", encoding="utf-8", newline="") as targetFile:
    writer = csv.DictWriter(targetFile, fieldnames=["id", "answer"])
    writer.writeheader()

    for rowIndex, row in enumerate(rowList, start=1):
        promptText = str(row["prompt"]).strip()
        promptMessages = [
            {"role": "system", "content": systemPrompt},
            {"role": "user", "content": promptText}
        ]

        fullPrompt = tokenizer.apply_chat_template(
            promptMessages,
            tokenize=False,
            add_generation_prompt=True
        )

        answerText = generateTextFunc(
            model,
            tokenizer,
            prompt=fullPrompt,
            max_tokens=maxNewTokens,
            sampler=sampler,
            logits_processors=logitsProcessors,
            verbose=False
        )
        answerText = cleanAnswer(answerText)

        writer.writerow({"id": row["id"], "answer": answerText})
        print(f"{rowIndex}/{len(rowList)}: {row['id']}")

print(f"prediction rows: {len(rowList)}")
print(f"saved submission: {predictionOutputPath.resolve()}")


Fetching 7 files: 100%|██████████| 7/7 [00:00<00:00, 13008.47it/s]


1/643: CORP-TAX-001
2/643: CORP-TAX-002
3/643: CORP-TAX-003
4/643: CORP-TAX-004
5/643: CORP-TAX-005
6/643: CORP-TAX-006
7/643: CORP-TAX-007
8/643: CORP-TAX-008
9/643: CORP-TAX-009
10/643: CORP-TAX-010
11/643: CORP-TAX-011
12/643: CORP-TAX-012
13/643: CORP-TAX-013
14/643: CORP-TAX-014
15/643: CORP-TAX-015
16/643: CORP-TAX-016
17/643: CORP-TAX-017
18/643: CORP-TAX-018
19/643: CORP-TAX-019
20/643: CORP-TAX-020
21/643: CORP-TAX-021
22/643: CORP-TAX-022
23/643: CORP-TAX-023
24/643: CORP-TAX-024
25/643: CORP-TAX-025
26/643: CORP-TAX-026
27/643: CORP-TAX-027
28/643: CORP-TAX-028
29/643: CORP-TAX-029
30/643: CORP-TAX-030
31/643: CORP-TAX-031
32/643: CORP-TAX-032
33/643: CORP-TAX-033
34/643: CORP-TAX-034
35/643: CORP-TAX-035
36/643: CORP-TAX-036
37/643: CORP-TAX-037
38/643: CORP-TAX-038
39/643: CORP-TAX-039
40/643: CORP-TAX-040
41/643: CORP-TAX-041
42/643: CORP-TAX-042
43/643: CORP-TAX-043
44/643: CORP-TAX-044
45/643: CORP-TAX-045
46/643: CORP-TAX-046
47/643: CORP-TAX-047
48/643: CORP-TAX-048
4